In [1]:
import numpy as np
import matplotlib.pyplot as plt

In [2]:
def dieukienbien(N_time,N_x,x_t0,t_x0,t_xl):
    u = np.zeros((N_x,N_time),dtype=float)

    #bien dau la t, bien thu 2 la x
    #x_t0 la tai thoi diem ban dau T = 100

    for i in range(N_time):
        u[N_x-1][i] = (t_xl)

    for i in range(N_time):
        u[0][i] = (t_x0)

    for i in range(N_x):
        u[i][0] = (x_t0)

    return u

In [3]:
def hamso(kappa,C, rho, N_x, N_time,l,t_max):
    h = l/(N_x-1)
    k = t_max/(N_time-1)
    alpha = np.sqrt(kappa/(C*rho))
    eta  = alpha ** 2 * k / h**2

    beta1 = eta/(2*eta+1)
    beta2 = 1/(2*eta+1)
    return beta1, beta2, h, k

In [4]:
def luufile(u, N_x, N_time, l, t_max,h,k,filename):
    with open(filename, "w", encoding="utf-8") as file:
            file.write("# Bai toan truyen nhiet 1D\n")
            file.write("# Phuong phap sai phan huu han hien\n")
            file.write("#\n")
            file.write(f"# N_x    = {N_x}\n")
            file.write(f"# N_time = {N_time}\n")
            file.write(f"# l      = {l}\n")
            file.write(f"# t_max  = {t_max}\n")
            file.write("#\n")
            file.write("# Cot du lieu: j   i   t   x   u\n")

            for j in range(N_time):
                t = j * k
                for i in range(N_x):
                    x = i * h
                    file.write(f"{j:5d} {i:5d} {t:15.8e} {x:15.8e} {u[i][j]:15.8e}\n")
                file.write("\n")

In [5]:
def tinhtoan_backward(u, kappa, C, rho, N_x, N_time, l, t_max, N_max,tol):
    beta1, beta2, h, k  = hamso(kappa,C, rho, N_x, N_time,l,t_max)
    filename = f"ketqua_hoitu.dat"
    with open(filename, "w", encoding="utf-8") as file:
        for j in range (1,N_time):
            err = np.zeros(N_x-2)
            for q in range (1,N_max):
                u_old = u.copy()
                for i in range(1,N_x-1):
                    u[i][j] = beta1 * (u[i+1][j] + u[i-1][j]) + beta2 * u[i][j-1]   
                    err[i-1] = abs(u[i][j] - u_old[i][j])
                if max(err) < tol:
                    file.write(f"Ket qua hoi tu tai vong lap thu: {q} tai buoc thoi gian thu: {j}.\n")
                    if np.mod(j, 50) == 0:
                        print(f"Ket qua hoi tu tai vong lap thu: {q} tai buoc thoi gian thu: {j}.\n")
                    break
                else:
                    continue
        luufile(u, N_x, N_time, l, t_max,h,k,filename="ketqua_truyen_nhiet.dat")
    return u

In [91]:
# Dieu kien bien

N_time = 600
N_x = 200
x_t0 = 100 #Do C
t_x0 = 0 #Do C
t_xl = 0 #Do C

# Tham so bai toan
kappa  = 210
C = 900
rho = 2700
l = 1
t_max = 1800

tol  = 1e-6
N_max = 10000

u = dieukienbien(N_time,N_x,x_t0,t_x0,t_xl)

In [92]:
solution  = tinhtoan_backward(u, kappa,C, rho, N_x, N_time,l,t_max, N_max,tol)

Ket qua hoi tu tai vong lap thu: 174 tai buoc thoi gian thu: 50.

Ket qua hoi tu tai vong lap thu: 173 tai buoc thoi gian thu: 100.

Ket qua hoi tu tai vong lap thu: 172 tai buoc thoi gian thu: 150.

Ket qua hoi tu tai vong lap thu: 171 tai buoc thoi gian thu: 200.

Ket qua hoi tu tai vong lap thu: 169 tai buoc thoi gian thu: 250.

Ket qua hoi tu tai vong lap thu: 168 tai buoc thoi gian thu: 300.

Ket qua hoi tu tai vong lap thu: 166 tai buoc thoi gian thu: 350.

Ket qua hoi tu tai vong lap thu: 165 tai buoc thoi gian thu: 400.

Ket qua hoi tu tai vong lap thu: 164 tai buoc thoi gian thu: 450.

Ket qua hoi tu tai vong lap thu: 162 tai buoc thoi gian thu: 500.

Ket qua hoi tu tai vong lap thu: 161 tai buoc thoi gian thu: 550.



## 2 thanh

In [9]:
def dieukienbien_2thanh(N_time, N_x, T_left, T_right, l_bar):
    l_total = 2.0 * l_bar
    x = np.linspace(0.0, l_total, N_x)

    u = np.zeros((N_x, N_time), dtype=float)

    # Dieu kien dau
    for i in range(N_x):
        if x[i] <= l_bar:
            u[i, 0] = T_left
        else:
            u[i, 0] = T_right

    # Dieu kien bien hai dau ngoai
    u[0, :] = 0.0
    u[-1, :] = 0.0

    # chac chan tai t = 0 hai bien van bang 0
    u[0, 0] = 0.0
    u[-1, 0] = 0.0

    return u, x

In [10]:
N_time = 600
N_x = 101          # lay le de x = 0.25 nam dung tren nut luoi

T_left = 100.0
T_right = 50.0

kappa = 210.0
C = 900.0
rho = 2700.0

l_bar = 0.25
l_total = 2.0 * l_bar     # 0.5 m

t_max = 1800.0

tol = 1e-6
N_max = 10000

In [14]:
def tinhtoan_backward_2thanh(u, kappa, C, rho, N_x, N_time, l, t_max, N_max, tol,
                      file_data="truyen_nhiet.dat", file_log="ketqua_hoitu.dat"):
    beta1, beta2, h, k = hamso(kappa, C, rho, N_x, N_time, l, t_max)

    with open(file_log, "w", encoding="utf-8") as file:
        for j in range(1, N_time):
            err = np.zeros(N_x - 2)

            for q in range(1, N_max + 1):
                u_old = u.copy()

                for i in range(1, N_x - 1):
                    u[i][j] = beta1 * (u[i+1][j] + u[i-1][j]) + beta2 * u[i][j-1]
                    err[i-1] = abs(u[i][j] - u_old[i][j])

                if np.max(err) < tol:
                    file.write(f"Ket qua hoi tu tai vong lap thu: {q} tai buoc thoi gian thu: {j}.\n")
                    if np.mod(j, 50) == 0:
                        print(f"Ket qua hoi tu tai vong lap thu: {q} tai buoc thoi gian thu: {j}.")
                    break
            else:
                file.write(f"Khong hoi tu tai buoc thoi gian thu: {j} sau {N_max} vong lap.\n")

    luufile(u, N_x, N_time, l, t_max, h, k, filename=file_data)
    return u

In [15]:
u_2thanh, x_2thanh = dieukienbien_2thanh(N_time, N_x, T_left, T_right, l_bar)

solution_2thanh = tinhtoan_backward_2thanh(
    u_2thanh,
    kappa, C, rho,
    N_x, N_time,
    l_total, t_max,
    N_max, tol,
    file_data="truyen_nhiet_2thanh.dat",
    file_log="ketqua_hoitu_2thanh.dat"
)

Ket qua hoi tu tai vong lap thu: 168 tai buoc thoi gian thu: 50.
Ket qua hoi tu tai vong lap thu: 162 tai buoc thoi gian thu: 100.
Ket qua hoi tu tai vong lap thu: 157 tai buoc thoi gian thu: 150.
Ket qua hoi tu tai vong lap thu: 152 tai buoc thoi gian thu: 200.
Ket qua hoi tu tai vong lap thu: 146 tai buoc thoi gian thu: 250.
Ket qua hoi tu tai vong lap thu: 141 tai buoc thoi gian thu: 300.
Ket qua hoi tu tai vong lap thu: 135 tai buoc thoi gian thu: 350.
Ket qua hoi tu tai vong lap thu: 130 tai buoc thoi gian thu: 400.
Ket qua hoi tu tai vong lap thu: 124 tai buoc thoi gian thu: 450.
Ket qua hoi tu tai vong lap thu: 119 tai buoc thoi gian thu: 500.
Ket qua hoi tu tai vong lap thu: 113 tai buoc thoi gian thu: 550.
